# Scaled Dot-Product Self-Attention Mechanism
## Chapter 3: Coding Attention Mechanisms From Scratch

The **self-attention mechanism** is the core computational block of Transformer models. It enables each token in an input sequence to dynamically compare itself against every other token, capturing contextual dependencies regardless of distance.

---

### Mathematical Formulation:
Given an input sequence representation $\mathbf{X} \in \mathbb{R}^{B \times T \times d_{in}}$:

1. **Linear Projections**:
   $$\mathbf{Q} = \mathbf{X} \mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X} \mathbf{W}_K, \quad \mathbf{V} = \mathbf{X} \mathbf{W}_V$$
   where $\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V \in \mathbb{R}^{d_{in} \times d_{out}}$.

2. **Attention Scores (Dot-Product Similarity)**:
   $$\mathbf{S} = \mathbf{Q} \mathbf{K}^\top \in \mathbb{R}^{B \times T \times T}$$

3. **Scaling & Softmax Normalization**:
   To prevent vanishing gradients for large dimensions $d_k$, scores are scaled by $\frac{1}{\sqrt{d_k}}$:
   $$\mathbf{A} = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d_k}}\right)$$

4. **Context Vectors (Weighted Value Aggregation)**:
   $$\mathbf{Z} = \mathbf{A} \mathbf{V} \in \mathbb{R}^{B \times T \times d_{out}}$$

---
## 1. Imports and Dependencies
We import PyTorch and essential modules for tensor manipulation and linear layers.

In [1]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

---
## 2. Implementing the `SelfAttention` Module

The `SelfAttention` class encapsulates:
- Trainable projection matrices $\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V$ implemented as `nn.Linear`.
- Dimension-scaled dot-product computation.
- Softmax normalization across the final sequence dimension.
- Matrix multiplication with values $\mathbf{V}$ to produce context vectors $\mathbf{Z}$.

In [ ]:
from torch import softmax
class SelfAttention(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()

        self.W_query = nn.Linear(d_in , d_out , bias = qkv_bias ) 
        self.W_key = nn.Linear(d_in, d_out , bias = qkv_bias )
        self.W_value = nn.Linear(d_in, d_out , bias = qkv_bias)

    def forward(self ,x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)o

        d_k = key.size(-1)

        atten_score = query @ key.transpose(-2 , -1)
        atten_weights = torch.softmax(atten_score / math.sqrt(d_k) , dim = -1)

        context = atten_weights @ value 

        return context 

---
## 3. Verification & Dimensional Analysis

Let's test `SelfAttention` on a batch:
- **Batch Size ($B$)**: 1
- **Sequence Length ($T$)**: 3
- **Input Dimension ($d_{in}$)**: 4
- **Output Projection Dimension ($d_{out}$)**: 2
- **Expected Output Shape**: $(B, T, d_{out}) = (1, 3, 2)$

In [3]:
d_in = 4
d_out = 2

torch.manual_seed(789)

input = torch.randn(1 , 3, d_in)

selfAttention = SelfAttention(d_in, d_out)

print(selfAttention(input))

tensor([[[-0.2125, -0.1429],
         [-0.1826, -0.1214],
         [-0.1563, -0.0989]]], grad_fn=<UnsafeViewBackward0>)
